# Wind & Solar Capacity Predictor

Target => val_geracaoverificada

In [2]:
import pandas as pd
import numpy as np
from os import listdir

Agregando os dados extraidos da ONS:

In [2]:
df = pd.concat(
    [pd.read_parquet(f"./database/{file}") for file in listdir("./database") if file.endswith('.parquet')],
    ignore_index=True
)

Ajustando a tipagem das colunas do DF de acordo com a documentação:

In [5]:
float_columns = [
    "val_latitudesecoletora",
    "val_longitudesecoletora",
    "val_latitudepontoconexao",
    "val_longitudepontoconexao",
    "val_geracaoprogramada",
    "val_geracaoverificada",
    "val_capacidadeinstalada",
    "val_fatorcapacidade"
]

df[float_columns] = df[float_columns].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

df["din_instante"] = pd.to_datetime(df["din_instante"])

In [ ]:
print(df.sort_values(by="din_instante").iloc[1]) # Encontra o registro mais antigo

id_subsistema                                     NE
nom_subsistema                              Nordeste
id_estado                                         CE
nom_estado                                     CEARA
cod_pontoconexao                         CESBT-230-A
nom_pontoconexao             SOBRAL III - 230 kV (A)
nom_localizacao                              Litoral
val_latitudesecoletora                     -2.919422
val_longitudesecoletora                    -41.03165
val_latitudepontoconexao                   -3.690358
val_longitudepontoconexao                 -40.300463
nom_modalidadeoperacao                        Tipo I
nom_tipousina                                 Eólica
nom_usina_conjunto                     Praia Formosa
id_ons                                         CEUFM
ceg                            EOL.CV.CE.028631-1.01
din_instante                     2009-07-01 01:00:00
val_geracaoprogramada                            0.0
val_geracaoverificada                         

Lê os dados metereológicos extraídos do (OpenMeteo)[https://open-meteo.com/] a partir de 2009 até 2026 (Novembro e Dezembro excluidos) com as coordenadas das capitais das unidades federativas do Brasil salvando todas as informações diárias disponíveis (inclusive as "opcionais"):

In [ ]:
def read_weather_csv(path: str) -> pd.DataFrame:
    meta = pd.read_csv(path, encoding="latin1", nrows=1)

    df = pd.read_csv(
        path,
        encoding="latin1",
        skiprows=3,
        parse_dates=["time"],
    )

    for col in meta.columns:
        df[col] = meta.at[0, col]

    return df


df_weather = pd.concat(
    [
        read_weather_csv(f"./database/{file}")
        for file in listdir("./database")
        if file.endswith(".csv")
    ],
    ignore_index=True,
)

print(df_weather.head())

        time  sunset (iso8601) sunrise (iso8601)  daylight_duration (s)  \
0 2009-01-01  2009-01-01T12:02  2009-01-01T04:17               27947.16   
1 2009-01-02  2009-01-02T12:04  2009-01-02T04:17               28025.71   
2 2009-01-03  2009-01-03T12:05  2009-01-03T04:16               28110.17   
3 2009-01-04  2009-01-04T12:06  2009-01-04T04:16               28200.37   
4 2009-01-05  2009-01-05T12:07  2009-01-05T04:16               28296.18   

   sunshine_duration (s)  precipitation_hours (h)  precipitation_sum (mm)  \
0                   0.00                      3.0                     0.4   
1               17208.15                      7.0                     2.0   
2               14441.04                      4.0                     1.2   
3                   0.00                     23.0                     8.2   
4               25200.00                      0.0                     0.0   

   rain_sum (mm)  snowfall_sum (cm)  et0_fao_evapotranspiration (mm)  ...  \
0        